In [1]:
import duckdb
import pandas as pd

print("=========================================================================")
print("🔄 ESTABLECIENDO CONEXIÓN DIRECTA Y CONFIGURANDO RUTAS")
print("=========================================================================")

# 1. Definimos las rutas absolutas exactas de tu Lakehouse (tomadas de tu Celda 1)
FACT_PATH = "/home/jcc/Proyectos/peru-budget-lakehouse/data/03_gold/fact_presupuesto.parquet"
DIM_INST_PATH = "/home/jcc/Proyectos/peru-budget-lakehouse/data/03_gold/dim_institucion.parquet"
SILVER_PATH = "/home/jcc/Proyectos/peru-budget-lakehouse/data/02_silver/mef_final_silver.parquet"

# 2. Inicializamos la conexión local en memoria aquí mismo
con = duckdb.connect()

print("🚀 INICIANDO AUDITORÍA QUIRÚRGICA: MISTERIO 2025 VS 2026")
print("=========================================================================")

# PRUEBA 1: Conteo físico directo en el archivo Parquet de la capa GOLD
print("\n🕵️‍♂️ [TEST 1] Conteo de filas y montos totales en la capa GOLD (Fact Table Parquet):")
try:
    df_fact = con.execute(f"""
        SELECT 
            ano_eje, 
            COUNT(*) as total_filas,
            SUM(monto) as suma_monto_total
        FROM '{FACT_PATH}'
        GROUP BY ano_eje
        ORDER BY ano_eje
    """).df()
    print(df_fact.to_string(index=False))
except Exception as e:
    print(f"Error en TEST 1: {e}")

# PRUEBA 2: Conteo físico directo en el archivo Parquet de la capa SILVER
print("\n🕵️‍♂️ [TEST 2] Conteo de filas y montos totales en la capa SILVER (Antes del modelo estrella):")
try:
    df_silver = con.execute(f"""
        SELECT 
            ano_eje, 
            COUNT(*) as total_filas,
            SUM(monto) as suma_monto_total
        FROM '{SILVER_PATH}'
        GROUP BY ano_eje
        ORDER BY ano_eje
    """).df()
    print(df_silver.to_string(index=False))
except Exception as e:
    print(f"Error en TEST 2: {e}")

# PRUEBA 3: Verificación de clonación de registros (Muestra de los montos más altos)
print("\n🕵️‍♂️ [TEST 3] Comparando los montos top por año para ver si las filas son clones:")
try:
    df_clon = con.execute(f"""
        WITH ranked_data AS (
            SELECT 
                ano_eje,
                monto,
                sk_institucion_id,
                ROW_NUMBER() OVER (PARTITION BY ano_eje ORDER BY monto DESC) as rn
            FROM '{FACT_PATH}'
        )
        SELECT ano_eje, rn, monto, sk_institucion_id
        FROM ranked_data
        WHERE rn <= 3
        ORDER BY rn, ano_eje
    """).df()
    print(df_clon.to_string(index=False))
except Exception as e:
    print(f"Error en TEST 3: {e}")

🔄 ESTABLECIENDO CONEXIÓN DIRECTA Y CONFIGURANDO RUTAS
🚀 INICIANDO AUDITORÍA QUIRÚRGICA: MISTERIO 2025 VS 2026

🕵️‍♂️ [TEST 1] Conteo de filas y montos totales en la capa GOLD (Fact Table Parquet):
 ano_eje  total_filas  suma_monto_total
    2022      8998862      1.507137e+12
    2023      9419339      1.600378e+12
    2024      9492993      1.717124e+12
    2025      9654223      1.811927e+12
    2026      9654223      1.811927e+12

🕵️‍♂️ [TEST 2] Conteo de filas y montos totales en la capa SILVER (Antes del modelo estrella):
 ano_eje  total_filas  suma_monto_total
    2022      8998862      1.507137e+12
    2023      9419339      1.600378e+12
    2024      9492993      1.717124e+12
    2025      9654223      1.811927e+12
    2026      9654223      1.811927e+12

🕵️‍♂️ [TEST 3] Comparando los montos top por año para ver si las filas son clones:
 ano_eje  rn        monto    sk_institucion_id
    2022   1 9.326233e+09 14346505403840840120
    2023   1 1.098984e+10 14346505403840840120
  